# Identifying compromised devices (indicators of compromise)  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 4.4.c: Weighted IOC-scoring engine

**Skill:** Detect Attacks · **Maps to:** 4.4.B, 4.4.C · **Language:** Python 3 (pytest)

## Problem
Each device under investigation reports a set of observed indicators of compromise (IOCs). Each
IOC type carries a fixed severity weight. Build the scoring engine that sums the weights of the
IOCs observed on a device and returns an overall compromise verdict.

## Function contract
Implement `def solve(iocs: list) -> str` in the code editor (see the Reference section below).

## Input
A list of strings, each an IOC type observed on one device. Duplicate entries in the list count
only once (an IOC type either is or is not present on the device).

## IOC weight table
| IOC type string | weight |
|---|---|
| `"known_malicious_hash"` | 6 |
| `"av_disabled"` | 5 |
| `"new_admin_account"` | 4 |
| `"unusual_outbound_traffic"` | 3 |
| `"unexpected_scheduled_task"` | 3 |
| `"registry_run_key_added"` | 2 |
| `"failed_login_cluster"` | 1 |

Any IOC type string not in this table contributes a weight of 0 (it is ignored, not an error).

## Output
One of the strings: `"clean"` (total score < 4), `"suspicious"` (4 <= total score < 8),
`"compromised"` (total score >= 8).

## Examples
```
["av_disabled", "new_admin_account"]
    -> "compromised"   # 5 + 4 = 9 >= 8

["failed_login_cluster"]
    -> "clean"          # 1 < 4

["new_admin_account", "failed_login_cluster"]
    -> "suspicious"      # 4 + 1 = 5, in [4, 8)
```

## Constraints
- Deterministic, offline, standard library only.
- Duplicate IOC types in the input list must not be double-counted.
- Unknown/unrecognized IOC type strings contribute 0 and must not raise an error.
- Thresholds are: `< 4` clean, `4 <= score < 8` suspicious, `>= 8` compromised.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(iocs: list) -> str
```

### Input (`iocs`)
A list of `str` IOC type identifiers observed on a single device. Duplicates may appear in the
list but each IOC type is counted at most once toward the total score.

### IOC weight table
```python
WEIGHTS = {
    "known_malicious_hash": 6,
    "av_disabled": 5,
    "new_admin_account": 4,
    "unusual_outbound_traffic": 3,
    "unexpected_scheduled_task": 3,
    "registry_run_key_added": 2,
    "failed_login_cluster": 1,
}
```
Any string not present in this table contributes a weight of `0`.

### Output (`result`)
One of `"clean"`, `"suspicious"`, `"compromised"`, based on the total score (sum of unique
matched weights):
- `score < 4` -> `"clean"`
- `4 <= score < 8` -> `"suspicious"`
- `score >= 8` -> `"compromised"`

### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only unless the hidden grading tests: allow_deps` lists otherwise.
- Duplicate IOC type strings in the input are de-duplicated before scoring.
- Unrecognized IOC strings do not raise; they simply add 0 to the score.
- Thresholds are as specified above (lower bound inclusive, upper bound exclusive per band).

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 4.4.c. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""

WEIGHTS = {
    "known_malicious_hash": 6,
    "av_disabled": 5,
    "new_admin_account": 4,
    "unusual_outbound_traffic": 3,
    "unexpected_scheduled_task": 3,
    "registry_run_key_added": 2,
    "failed_login_cluster": 1,
}


def solve(iocs: list) -> str:
    """Sum the weights of unique observed IOCs and classify a compromise verdict.

    Args:
        iocs: list of str IOC type identifiers observed on one device.

    Returns:
        "clean" (score < 4), "suspicious" (4 <= score < 8), or "compromised" (score >= 8).
    """
    unique_iocs = set(iocs)
    score = sum(WEIGHTS.get(ioc, 0) for ioc in unique_iocs)

    if score >= 8:
        return "compromised"
    if score >= 4:
        return "suspicious"
    return "clean"


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_example_1_compromised():
    assert impl.solve(["av_disabled", "new_admin_account"]) == "compromised"


def test_example_2_clean():
    assert impl.solve(["failed_login_cluster"]) == "clean"


def test_example_3_suspicious():
    assert impl.solve(["new_admin_account", "failed_login_cluster"]) == "suspicious"


# ---- hidden / edge cases -------------------------------------------------
def test_edge_empty_list_is_clean():
    assert impl.solve([]) == "clean"


def test_unknown_ioc_contributes_zero():
    assert impl.solve(["unrecognized_ioc_type"]) == "clean"


def test_duplicate_iocs_not_double_counted():
    # av_disabled alone = 5 (suspicious). If duplicated and double-counted it would
    # wrongly become 10 (compromised).
    assert impl.solve(["av_disabled", "av_disabled", "av_disabled"]) == "suspicious"


def test_boundary_score_exactly_four_is_suspicious():
    assert impl.solve(["new_admin_account"]) == "suspicious"


def test_boundary_score_just_below_four_is_clean():
    assert impl.solve(["unusual_outbound_traffic"]) == "clean"


def test_boundary_score_exactly_eight_is_compromised():
    assert impl.solve(["av_disabled", "unusual_outbound_traffic"]) == "compromised"


def test_boundary_score_seven_is_suspicious():
    assert impl.solve(["unusual_outbound_traffic", "unexpected_scheduled_task",
                        "failed_login_cluster"]) == "suspicious"


def test_high_severity_combination_is_compromised():
    assert impl.solve(["known_malicious_hash", "unusual_outbound_traffic",
                        "registry_run_key_added"]) == "compromised"


def test_all_iocs_combined_is_compromised():
    all_iocs = ["known_malicious_hash", "av_disabled", "new_admin_account",
                "unusual_outbound_traffic", "unexpected_scheduled_task",
                "registry_run_key_added", "failed_login_cluster"]
    assert impl.solve(all_iocs) == "compromised"


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
